# Probabilistic Machine Learning Practice Notebook

This notebook is a hands-on companion to the Markdown file on **Probabilistic Machine Learning**.
It demonstrates key methods that produce probabilistic outputs rather than point predictions.

Topics covered:

1. Naive Bayes classification
2. Gaussian Mixture Models (GMM)
3. EM-style clustering intuition (soft assignments)
4. Gaussian Process regression
5. Generative vs discriminative model comparison
6. Probability calibration
7. Summary table
8. Mini exercises

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, brier_score_loss
np.random.seed(42)

## 1. Naive Bayes Classification

Naive Bayes is a **generative classifier** that models the joint P(X, Y) by assuming
features are conditionally independent given the class:

$$P(Y|X) \propto P(Y) \prod_{j} P(X_j|Y)$$

Despite the strong independence assumption, it often works well in practice.

In [ ]:
X, y = make_classification(n_samples=300, n_features=6, n_informative=4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

nb = GaussianNB().fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
y_prob_nb = nb.predict_proba(X_test)[:, 1]

pd.DataFrame({
    'Metric': ['Accuracy', 'Brier score'],
    'Value':  [accuracy_score(y_test, y_pred_nb), brier_score_loss(y_test, y_prob_nb)]
})

## 2. Gaussian Mixture Models (GMM)

A GMM models data as a mixture of K Gaussians:

$$P(X) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(X;\, \mu_k, \Sigma_k)$$

Parameters are fit with the **Expectation-Maximization (EM)** algorithm.

In [ ]:
X_blobs, _ = make_blobs(n_samples=250, centers=3, cluster_std=1.2, random_state=42)
gmm = GaussianMixture(n_components=3, random_state=42)
gmm.fit(X_blobs)
labels = gmm.predict(X_blobs)
probs  = gmm.predict_proba(X_blobs)

pd.DataFrame({'Component': [1, 2, 3], 'Weight': gmm.weights_})

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='tab10', s=20)
plt.scatter(gmm.means_[:, 0], gmm.means_[:, 1], marker='X', s=200, c='black', label='Component means')
plt.title('Gaussian Mixture Model Clustering')
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend()
plt.show()

## 3. EM-Style Clustering: Soft Assignments

Unlike k-means (hard assignment), GMM assigns each point fractional membership probabilities.

These responsibilities r_{ik} = P(component k | x_i) tell how likely each component
generated a given observation.

In [ ]:
resp_df = pd.DataFrame(probs[:8], columns=['Comp 1', 'Comp 2', 'Comp 3']).round(4)
resp_df['Hard assignment'] = labels[:8]
resp_df

**Observation:** Points near cluster boundaries have fractional membership, reflecting genuine ambiguity.

## 4. Gaussian Process Regression

A Gaussian Process (GP) is a distribution over functions.
After observing data, the GP posterior gives a **mean prediction plus a principled uncertainty band**:

$$f^* | X^*, X, y \sim \mathcal{N}(\mu^*, \Sigma^*)$$

The uncertainty band widens away from training data — this is epistemic uncertainty.

In [ ]:
X_gp    = np.atleast_2d(np.linspace(0, 10, 25)).T
y_gp    = np.sin(X_gp).ravel() + np.random.normal(0, 0.15, X_gp.shape[0])
X_star  = np.atleast_2d(np.linspace(-1, 12, 250)).T   # extend beyond training range

kernel = C(1.0) * RBF(length_scale=1.0)
gpr    = GaussianProcessRegressor(kernel=kernel, alpha=0.1**2, random_state=42)
gpr.fit(X_gp, y_gp)
mean_pred, std_pred = gpr.predict(X_star, return_std=True)

pd.DataFrame({
    'Metric': ['Train points', 'Mean predictive std (in-range)', 'Kernel after fit'],
    'Value':  [len(X_gp), round(std_pred[(X_star[:, 0] >= 0) & (X_star[:, 0] <= 10)].mean(), 4),
               str(gpr.kernel_)]
})

In [ ]:
plt.figure(figsize=(9, 4))
plt.scatter(X_gp[:, 0], y_gp, label='Observed', zorder=5)
plt.plot(X_star[:, 0], mean_pred, label='GP mean')
plt.fill_between(X_star[:, 0], mean_pred - 2*std_pred, mean_pred + 2*std_pred,
                 alpha=0.3, label='Mean +/- 2 std')
plt.title('Gaussian Process Regression (uncertainty widens outside training range)')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

## 5. Generative vs Discriminative Model Comparison

- **Generative (Naive Bayes):** Models P(X, Y) = P(X|Y) P(Y)
- **Discriminative (Logistic Regression):** Models P(Y|X) directly

Generative models can work with less data; discriminative models often have higher asymptotic accuracy.

In [ ]:
logreg    = LogisticRegression(max_iter=1000).fit(X_train, y_train)
y_pred_lr = logreg.predict(X_test)
y_prob_lr = logreg.predict_proba(X_test)[:, 1]

pd.DataFrame({
    'Model':    ['Naive Bayes (generative)', 'Logistic Regression (discriminative)'],
    'Accuracy': [accuracy_score(y_test, y_pred_nb), accuracy_score(y_test, y_pred_lr)],
    'Brier':    [brier_score_loss(y_test, y_prob_nb), brier_score_loss(y_test, y_prob_lr)]
})

## 6. Probability Calibration

A well-calibrated model outputs probabilities that match the true frequency of events.
A calibration curve plots predicted probability (x-axis) vs observed positive fraction (y-axis).

A perfectly calibrated model follows the diagonal.

In [ ]:
frac_pos_nb, mean_pred_nb = calibration_curve(y_test, y_prob_nb, n_bins=10)
frac_pos_lr, mean_pred_lr = calibration_curve(y_test, y_prob_lr, n_bins=10)

plt.figure(figsize=(6, 5))
plt.plot(mean_pred_nb, frac_pos_nb, marker='o', label='Naive Bayes')
plt.plot(mean_pred_lr, frac_pos_lr, marker='s', label='Logistic Regression')
plt.plot([0, 1], [0, 1], linestyle='--', label='Perfect calibration')
plt.title('Calibration Curves')
plt.xlabel('Predicted probability')
plt.ylabel('Observed positive fraction')
plt.legend()
plt.show()

## 7. Summary Table

In [ ]:
summary = pd.DataFrame({
    'Measure': [
        'NB accuracy', 'NB Brier score',
        'GMM components', 'GMM dominant weight',
        'GP mean predictive std',
        'LR accuracy', 'LR Brier score'
    ],
    'Value': [
        round(accuracy_score(y_test, y_pred_nb), 4),
        round(brier_score_loss(y_test, y_prob_nb), 4),
        gmm.n_components,
        round(gmm.weights_.max(), 4),
        round(std_pred.mean(), 4),
        round(accuracy_score(y_test, y_pred_lr), 4),
        round(brier_score_loss(y_test, y_prob_lr), 4)
    ]
})
summary

## 8. Mini Exercises

Try these on your own:

1. Reduce the training set size to 50 samples and compare Naive Bayes vs Logistic Regression accuracy — does the generative model hold up better with less data?
2. Change GMM to 2 or 5 components and compare the BIC (Bayesian Information Criterion) scores to find the best number of clusters.
3. In the GP regression cell, change the kernel to Matern or increase the noise level and compare uncertainty bands.
4. Extend the GP to training points spaced unevenly and inspect where uncertainty grows.
5. Apply probability calibration (sklearn.calibration.CalibratedClassifierCV) to Naive Bayes and compare the calibration curve before and after.
6. Replace the synthetic classification dataset with an imbalanced one and compare Brier scores carefully.
7. Compare the GMM log-likelihood for different numbers of components and plot a BIC elbow curve.
8. Generate data from a mixture of two Gaussians and verify that GMM recovers the true component means and weights.

These exercises are especially useful for AI, uncertainty-aware ML, surrogate modeling, and probabilistic forecasting.